In [ ]:
print("Hello Rui")

Hello Rui


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/train.csv")

print("Shape:", train.shape)
print("\nColumns:", train.columns.tolist())
print("\nData Types:\n", train.dtypes)
print("\nFirst 3 rows:\n", train.head(3))
print("\nMissing Values:\n", train.isnull().sum())
print("\nLabel distribution:\n", train['label'].value_counts().sort_index())
print("\nSample 'comment' values:\n", train['comment'].dropna().head(3).tolist())
print("\nNumerical columns stats:\n", train[["post_id", "emoticon_1", "emoticon_2", "emoticon_3", "upvote", "downvote", "if_1", "if_2"]].describe())
print("\nCategorical unique values:")
for col in ["race", "religion", "gender", "disability"]:
    print(f"  {col}: {train[col].nunique()} unique →", train[col].unique()[:5])

Shape: (198000, 15)

Columns: ['created_date', 'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender', 'disability', 'comment', 'label']

Data Types:
 created_date    object
post_id          int64
emoticon_1       int64
emoticon_2       int64
emoticon_3       int64
upvote           int64
downvote         int64
if_1             int64
if_2             int64
race            object
religion        object
gender          object
disability        bool
comment         object
label            int64
dtype: object

First 3 rows:
                        created_date  post_id  emoticon_1  emoticon_2  \
0  2024-01-18 08:43:57.397508+00:00       73           0           0   
1  2024-03-24 21:43:11.490017+00:00       39           0           0   
2  2024-04-24 20:32:17.014931+00:00       31           0           1   

   emoticon_3  upvote  downvote  if_1  if_2 race religion gender  disability  \
0           0       0         1     0   

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

train = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/train.csv")
train['comment'] = train['comment'].fillna('')

X = train.drop(columns=['label'])
y = train['label']

# Part A: Stratified
X_train_A, X_val_A, y_train_A, y_val_A = train_test_split(
    X, y, test_size=0.4, random_state=2306, stratify=y
)
train_counts_A = np.array(y_train_A.value_counts().sort_index())
val_counts_A   = np.array(y_val_A.value_counts().sort_index())
print("Q1A - Train counts (stratified):", train_counts_A)
print("Q1A - Val counts   (stratified):", val_counts_A)

# Part B: Non-Stratified
X_train_B, X_val_B, y_train_B, y_val_B = train_test_split(
    X, y, test_size=0.4, random_state=2306, stratify=None
)
train_counts_B = np.array(y_train_B.value_counts().sort_index())
val_counts_B   = np.array(y_val_B.value_counts().sort_index())
print("Q1B - Train counts (non-stratified):", train_counts_B)
print("Q1B - Val counts   (non-stratified):", val_counts_B)

# Part C: Max absolute difference in proportions
prop_A = val_counts_A / val_counts_A.sum()
prop_B = val_counts_B / val_counts_B.sum()
max_diff = np.max(np.abs(prop_A - prop_B))
print(f"\nQ1C - Max absolute difference: {max_diff:.4f}")

Q1A - Train counts (stratified): [68504  9551 37464  3281]
Q1A - Val counts   (stratified): [45669  6367 24976  2188]
Q1B - Train counts (non-stratified): [68491  9599 37444  3266]
Q1B - Val counts   (non-stratified): [45682  6319 24996  2203]

Q1C - Max absolute difference: 0.0006


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

train = pd.read_csv("/kaggle/input/competitions/comment-category-prediction-challenge/train.csv")
train['comment'] = train['comment'].fillna('')
X = train.drop(columns=['label'])
y = train['label']

# Stratified split
X_train_A, X_val_A, y_train_A, y_val_A = train_test_split(
    X, y, test_size=0.4, random_state=2306, stratify=y
)

# Step 1: Drop created_date
x_train = X_train_A.copy().drop(columns=['created_date'])
x_test  = X_val_A.copy().drop(columns=['created_date'])
y_train = y_train_A.copy()
y_test  = y_val_A.copy()

# Step 2: Extract text
text_x_train = x_train['comment']
text_x_test  = x_test['comment']
x_train = x_train.drop(columns=['comment'])
x_test  = x_test.drop(columns=['comment'])

# Step 3: ColumnTransformer
cat_cols = ["race", "religion", "gender", "disability"]
num_cols = ["post_id", "emoticon_1", "emoticon_2", "emoticon_3",
            "upvote", "downvote", "if_1", "if_2"]

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ],
    remainder='passthrough'
)

x_train_tabular = preprocessor.fit_transform(x_train)
x_test_tabular  = preprocessor.transform(x_test)

# Step 4: Text Cleaning — NOTE: double space as per question
def normalize_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', '  ', text).strip()
    return text

text_x_train_norm = text_x_train.apply(normalize_text)
text_x_test_norm  = text_x_test.apply(normalize_text)

# Step 5: TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tf_idf_train = tfidf.fit_transform(text_x_train_norm)
tf_idf_test  = tfidf.transform(text_x_test_norm)

# Step 6: Combine
X_train_final = hstack([x_train_tabular, tf_idf_train])
X_test_final  = hstack([x_test_tabular, tf_idf_test])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"Q2 - Sum of all values in X_train_final: {X_train_final.sum():.3f}")

X_train_final shape: (118800, 5029)
Q2 - Sum of all values in X_train_final: 904262.933


In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# SVD — fit only on X_train_final
svd = TruncatedSVD(n_components=300, random_state=2306)
X_train_reduced = svd.fit_transform(X_train_final)
X_test_reduced  = svd.transform(X_test_final)

print(f"X_train_reduced shape: {X_train_reduced.shape}")
print(f"X_test_reduced shape:  {X_test_reduced.shape}")

# Q3 — RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [5, 10, 15]
}

rf = RandomForestClassifier(random_state=2306)

randomized_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=5,
    cv=3,
    random_state=2306,
    n_jobs=-1
)

randomized_search.fit(X_train_reduced, y_train)

print(f"\nQ3 - Best params: {randomized_search.best_params_}")
print(f"Q3 - Best n_estimators: {randomized_search.best_params_['n_estimators']}")

X_train_reduced shape: (118800, 300)
X_test_reduced shape:  (79200, 300)

Q3 - Best params: {'n_estimators': 200, 'max_depth': 15}
Q3 - Best n_estimators: 200


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# Q4 - AdaBoost
ada = AdaBoostClassifier(n_estimators=50, random_state=2306)
ada.fit(X_train_reduced, y_train)

variance = np.var(ada.estimator_errors_)
print(f"Estimator errors length: {len(ada.estimator_errors_)}")
print(f"Q4 - Variance of estimator_errors_: {round(variance, 4)}")

Estimator errors length: 50
Q4 - Variance of estimator_errors_: 0.0058


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Q5 - Feature Importance
rf5 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2306)
rf5.fit(X_train_reduced, y_train)

max_idx = np.argmax(rf5.feature_importances_)
print(f"Q5 - Index of max feature importance: {max_idx}")
print(f"Q5 - Max importance value: {rf5.feature_importances_[max_idx]:.6f}")


Q5 - Index of max feature importance: 4
Q5 - Max importance value: 0.170272


In [ ]:
N = X_train_reduced.shape[1]  # 300
layers = [N, 128, 64, 32, 4]
total_weights = sum(layers[i] * layers[i+1] for i in range(len(layers)-1))
print(f"N = {N}")
print(f"Q6 - Total weights (no biases): {total_weights}")

N = 300
Q6 - Total weights (no biases): 48768


In [ ]:
from sklearn.neural_network import MLPClassifier

# Q7 - MLP with adam, 5 iterations
mlp_q7 = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=5,
    batch_size=32,
    random_state=2306
)
mlp_q7.fit(X_train_reduced, y_train)

print(f"Q7 - model.loss_: {round(mlp_q7.loss_, 4)}")

Q7 - model.loss_: 0.3271


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
from sklearn.metrics import f1_score

# Q8 - Two MLPs
mlp_A = MLPClassifier(hidden_layer_sizes=(100,), max_iter=5, random_state=2306, alpha=0.0001)
mlp_B = MLPClassifier(hidden_layer_sizes=(100,), max_iter=5, random_state=2306, alpha=1.0)

mlp_A.fit(X_train_reduced, y_train)
mlp_B.fit(X_train_reduced, y_train)

f1_A = f1_score(y_train, mlp_A.predict(X_train_reduced), average='macro')
f1_B = f1_score(y_train, mlp_B.predict(X_train_reduced), average='macro')

print(f"Q8 - F1 Model A (alpha=0.0001): {f1_A:.4f}")
print(f"Q8 - F1 Model B (alpha=1.0):    {f1_B:.4f}")
print(f"Q8 - Absolute difference:        {abs(f1_A - f1_B):.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


Q8 - F1 Model A (alpha=0.0001): 0.7498
Q8 - F1 Model B (alpha=1.0):    0.6171
Q8 - Absolute difference:        0.1327


In [ ]:
from sklearn.metrics import confusion_matrix

# Q9 - uses mlp_q7 trained in Q7 on validation set
y_pred_val = mlp_q7.predict(X_test_reduced)
cm = confusion_matrix(y_test, y_pred_val)

off_diagonal_sum = cm.sum() - np.trace(cm)
misclassification_rate = off_diagonal_sum / len(y_test)

print(f"Q9 - Confusion Matrix:\n{cm}")
print(f"Q9 - Off-diagonal sum: {off_diagonal_sum}")
print(f"Q9 - Total samples: {len(y_test)}")
print(f"Q9 - Misclassification rate: {round(misclassification_rate, 4)}")

Q9 - Confusion Matrix:
[[43266   331  1931   141]
 [  112  4670  1501    84]
 [ 1672  1370 21410   524]
 [   83   122  1049   934]]
Q9 - Off-diagonal sum: 8920
Q9 - Total samples: 79200
Q9 - Misclassification rate: 0.1126
